# 1. Base Setup

## 1.1 Install packages

In [51]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost

## 1.2 Load all needed imports

In [53]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [54]:
import cf_copilot
print(cf_copilot.__file__)

/Users/anton/code/DERNTOAN/cf_copilot/cf_copilot/__init__.py


In [55]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [56]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.8)

train_df = big_df[big_df["reference_date"] <= cutoff_date]
test_df = big_df[big_df["reference_date"] > cutoff_date]

X_train, y_train = preprocess(train_df)
X_test, y_test = preprocess(test_df)

Loading local file: /Users/anton/code/DERNTOAN/cf_copilot/raw_data/dataset.csv
Saved model_df (39152 rows)
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [57]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [58]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Linear models
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import RidgeClassifier

# SVM
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# Neighbors
from sklearn.neighbors import KNeighborsClassifier

# Naive Bayes
from sklearn.naive_bayes import GaussianNB

# Neural network
from sklearn.neural_network import MLPClassifier

# Single tree (useful as baseline)
from sklearn.tree import DecisionTreeClassifier


# Replace with your classifier
classifier = CatBoostClassifier()

In [59]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])

In [60]:
# from sklearn.preprocessing import LabelEncoder
# # This is XGBoost specific
# # le = LabelEncoder()
# # y_train_enc = le.fit_transform(y_train)
# # y_test_enc  = le.transform(y_test)

# pipeline.fit(X_train, y_train)

# base_probas = pipeline.predict_proba(X_test)
# base_preds = pipeline.predict(X_test)

In [61]:
def evaluate_model(probas, preds):
    return log_loss(y_test, probas)

In [67]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from cf_copilot.ml_logic.evaluation import evaluate_training_run
from cf_copilot.cashflow_prediction.evaluation import evaluate_forecast_holdout
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

models = {
    "RandomForest": RandomForestClassifier(),
    "ExtraTrees": ExtraTreesClassifier(),
    "HistGradientBoosting": HistGradientBoostingClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "LightGBM": LGBMClassifier(),
    "LogisticRegression": LogisticRegression(),
    "CatBoost": CatBoostClassifier()
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    try:
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
        pipe.fit(X_train, y_train_enc)

        probas = pipe.predict_proba(X_test)
        preds = pipe.predict(X_test)
        forecast, _ = evaluate_forecast_holdout(pipe, test_df)
        results.append({
            "model": name,
            "log_loss": evaluate_model(probas, preds),
            "f1_macro": f1_score(y_test_enc, preds, average="macro"),
            "accuracy": accuracy_score(y_test_enc, preds),
        })
        for key, value in forecast.items():
            results[-1][key] = value
    except Exception as e:
        print(f"  {name} failed: {e}")

results_df = pd.DataFrame(results).sort_values("log_loss")
results_df

Training RandomForest...

Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2538733.95
✅ Forecast MAPE weekly: 0.7818
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17930964.28
✅ Forecast total cf difference: -12502587.14
Training ExtraTrees...

Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2663606.24
✅ Forecast MAPE weekly: 0.8473
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17982247.08
✅ Forecast total cf difference: -12451304.34
Training HistGradientBoosting...

Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2631977.08
✅ Forecast MAPE weekly: 0.7882
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17159868.48
✅ Forecast total cf difference: -13273682.94
Training GradientBoosting...

Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2875893.95
✅ Forecast MAPE weekly: 1.0050
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17250971.19
✅ Forecast tota

/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Evaluating forecast on 21811 rows...


/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-pack

✅ Forecast MAE weekly: 2642061.31
✅ Forecast MAPE weekly: 0.7855
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17208176.48
✅ Forecast total cf difference: -13225374.94
Training LogisticRegression...


/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2639745.42
✅ Forecast MAPE weekly: 0.8382
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17553934.72
✅ Forecast total cf difference: -12879616.70
Training CatBoost...
Learning rate set to 0.098707
0:	learn: 1.7061028	total: 24.2ms	remaining: 24.2s
1:	learn: 1.5493988	total: 42.2ms	remaining: 21s
2:	learn: 1.4350199	total: 61.7ms	remaining: 20.5s
3:	learn: 1.3515274	total: 79.9ms	remaining: 19.9s
4:	learn: 1.2817487	total: 99.7ms	remaining: 19.8s
5:	learn: 1.2253885	total: 121ms	remaining: 20.1s
6:	learn: 1.1766704	total: 142ms	remaining: 20.1s
7:	learn: 1.1352156	total: 162ms	remaining: 20.1s
8:	learn: 1.0995943	total: 181ms	remaining: 19.9s
9:	learn: 1.0675403	total: 199ms	remaining: 19.7s
10:	learn: 1.0405168	total: 222ms	remaining: 19.9s
11:	learn: 1.0165264	total: 241ms	remaining: 19.9s
12:	learn: 0.9931557	total: 259ms	remaining: 19.7s
13:	learn: 0.9746327	total: 276ms	remaining: 19.4s
14:	learn: 0

,model,log_loss,f1_macro,accuracy,forecast_mae_weekly,forecast_mape_weekly,forecast_total_actual_cf,forecast_total_forecast_cf,forecast_total_cf_difference
7,CatBoost,0.755908,0.537333,0.741231,2.648957e+06,0.801364,3.043355e+07,1.687611e+07,-1.355744e+07
5,LightGBM,0.759118,0.546130,0.741094,2.642061e+06,0.785455,3.043355e+07,1.720818e+07,-1.322537e+07
2,HistGradientBoosting,0.767860,0.540598,0.739443,2.631977e+06,0.788182,3.043355e+07,1.715987e+07,-1.327368e+07
3,GradientBoosting,0.798023,0.487789,0.712622,2.875894e+06,1.005000,3.043355e+07,1.725097e+07,-1.318258e+07
4,XGBoost,0.819914,0.532729,0.734492,2.619138e+06,0.802727,3.043355e+07,1.700815e+07,-1.342541e+07
0,RandomForest,1.123282,0.550108,0.743249,2.538734e+06,0.781818,3.043355e+07,1.793096e+07,-1.250259e+07
1,ExtraTrees,1.182880,0.541717,0.739856,2.663606e+06,0.847273,3.043355e+07,1.798225e+07,-1.245130e+07
6,LogisticRegression,1.321874,0.087686,0.442254,2.639745e+06,0.838182,3.043355e+07,1.755393e+07,-1.287962e+07
